In [1]:
import pandas as pd
import numpy as np
import os

## 1. Origen de Viajeros, datos oficiales

La base de datos originales se desgarcó desde la Encuesta de ocupación hotelera, del Instituto Nacional de Estadística de España (INE)

Enlace: https://www.ine.es/jaxiT3/Tabla.htm?t=2038&L=0

Fecha de desgarcar: 11 de febrero de 2026

Titulo de la base de datos "Viajeros y pernoctaciones según país de residencia del viajero"

Nombre original automáticamente de la tabla desgarcarda "2308"

In [19]:
# Cargar datos
ruta_origen = os.path.join("..", 'Data', 'INEViajeroOrigenRaw11022026.csv')

origen = pd.read_csv(ruta_origen, sep=';')

In [20]:
# Transformar columns
origen['Total'] = origen['Total'].str.replace('.','')
origen['Total'] = origen['Total'].apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.int64)

origen['Países'] = origen['Países'].fillna(origen['RESIDENCIA/ORIGEN'])

In [21]:
# Filtrar los registros que necesitamos
origen = origen[origen['Viajeros y pernoctaciones'] == 'Viajero']
origen = origen[origen['Periodo'].str.contains('2021|2022')]
origen = origen[origen['Países'].str.contains('Total|UE27|UE28') == False]

In [22]:
# Quitar columnas no necesarias
origen = origen.drop(columns=['RESIDENCIA/ORIGEN', 'Viajeros y pernoctaciones'])

In [23]:
# cambiar nombres de columnas
origen = origen.rename(columns = {"Países": "pais",
                                  "Periodo": "periodo",
                                  "Total": "viajeros"})

In [24]:
# crear clumnas de año y mes
origen['año'] = pd.to_datetime(origen['periodo'].str.replace('M','-'), errors='coerce').dt.to_period('Y')
origen['mes'] = pd.to_datetime(origen['periodo'].str.replace('M','-'), errors='coerce').dt.month_name(locale = 'Spanish')

In [25]:
# crear la columna de origen de viajeros por zonas grandes


# Funcion que define las categorias (zonas) de origen
def classify_origin(label):
    # Residentes en España
    if label == 'Residentes en España':
        return "España"
    
    # Residentes de todos otros países (incluido todas las categorias seguientes)
    if label == 'Residentes en el Extranjero':
        return "El Extranjero"
    
    # Union Europea (UE27 post-2020)
    eu_list = ['Alemania', 'Austria', 'Bélgica', 'Dinamarca', 'Finlandia',
               'Francia', 'Grecia', 'Irlanda', 'Italia', 'Luxemburgo',
               'Países Bajos', 'Polonia', 'Portugal', 'República Checa', 'Suecia',
               'Resto de la U.E.']
    if label in eu_list:
        return "Unión Europea"
    
    # Resto de Europa (Non-EU)
    # Russia es incluido desde marzo de 2022, según las notas de INE
    resto_europa_list = ['Noruega', 'Reino Unido', 'Rusia', 'Suiza', 'Otros Países Europeos']
    if label in resto_europa_list:
        return "Resto de Europa"
    
    # Todos otros
    return "Resto del Mundo"


# Aplicar la funcion
origen['origen'] = origen['pais'].apply(classify_origin)

In [26]:
origen.info()

<class 'pandas.core.frame.DataFrame'>
Index: 696 entries, 684 to 20147
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype        
---  ------    --------------  -----        
 0   pais      696 non-null    object       
 1   periodo   696 non-null    object       
 2   viajeros  696 non-null    int64        
 3   año       696 non-null    period[Y-DEC]
 4   mes       696 non-null    object       
 5   origen    696 non-null    object       
dtypes: int64(1), object(4), period[Y-DEC](1)
memory usage: 38.1+ KB


In [27]:
# --- EXPORTACIÓN DEL DATASET LIMPIO ---

# 1. Nombre del nuevo archivo
nombre_origen_limpio = 'INEViajeroOrigenClean11022026.csv'

# 2. Construimos la ruta apuntando a la misma carpeta 'Data'
# Usamos '..' para subir un nivel y luego entrar en 'Data'
ruta_origen_guardado = os.path.join('..', 'Data', nombre_origen_limpio)

# 3. Guardamos el DataFrame
# index=False evita que se cree una columna extra de números
# encoding='utf-8-sig' 
origen.to_csv(ruta_origen_guardado, index=False, encoding='utf-8-sig')

## 2. Destino de Viajeros, datos oficiales

La base de datos originales se desgarcó desde la Encuesta de ocupación hotelera, del Instituto Nacional de Estadística de España (INE)

Enlace: https://www.ine.es/jaxiT3/Tabla.htm?t=2074&L=0

Fecha de desgarcar: 11 de febrero de 2026

Titulo de la base de datos "Viajeros y pernoctaciones por comunidades autónomas y provincias"

Nombre original automáticamente de la tabla desgarcarda "2074"

In [28]:
# Cargar datos
ruta_destino = os.path.join("..", 'Data', 'INEViajeroDestinoRaw11022026.csv')

destino = pd.read_csv(ruta_destino, sep=';')

In [29]:
# Transformar columns
destino['Total'] = destino['Total'].str.replace('.','')
destino['Total'] = destino['Total'].apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.int64)

# quitar los numeros de los nombres de comunidades
destino['Comunidades y Ciudades Autónomas'] = destino['Comunidades y Ciudades Autónomas'].str.replace(r'^\d+\s', '', regex=True)

# cambiar los valores de origen para que sean más cortos
destino['Residencia: Nivel 2'] = destino['Residencia: Nivel 2'].str.replace('Residentes en España', 'España')
destino['Residencia: Nivel 2'] = destino['Residencia: Nivel 2'].str.replace('Residentes en el Extranjero', 'El Extranjero')

In [30]:
# Filtrar los registros que necesitamos
destino = destino[destino['Viajeros y pernoctaciones'] == 'Viajero']
destino = destino[destino['Periodo'].str.contains('2021|2022')]
destino = destino[destino['Residencia: Nivel 2'].notna()] # quitar los de todos viajeros
destino = destino[destino['Comunidades y Ciudades Autónomas'].notna()] # quitar los de todos viajeros
destino = destino[destino['Provincias'].isna()] # solo quedamos con el nivel de viajeros totales de cada comunidad sin separar por provincia

In [31]:
# Quitar columnas no necesarias
destino = destino.drop(columns=['Totales Territoriales', 'Provincias', 'Viajeros y pernoctaciones', 'Residencia: Nivel 1'])

In [32]:
# cambiar nombres de columnas
destino = destino.rename(columns = {"Comunidades y Ciudades Autónomas": "comunidad",
                                    "Residencia: Nivel 2": "origen",
                                    "Periodo": "periodo",
                                    "Total": "viajeros"})

In [33]:
# crear clumnas de año y mes
destino['año'] = pd.to_datetime(destino['periodo'].str.replace('M','-'), errors='coerce').dt.to_period('Y')
destino['mes'] = pd.to_datetime(destino['periodo'].str.replace('M','-'), errors='coerce').dt.month_name(locale = 'Spanish')

In [34]:
# --- EXPORTACIÓN DEL DATASET LIMPIO ---

# 1. Nombre del nuevo archivo
nombre_destino_limpio = 'INEViajeroDestinoClean11022026.csv'

# 2. Construimos la ruta apuntando a la misma carpeta 'Data'
# Usamos '..' para subir un nivel y luego entrar en 'Data'
ruta_destino_guardado = os.path.join('..', 'Data', nombre_destino_limpio)

# 3. Guardamos el DataFrame
# index=False evita que se cree una columna extra de números
# encoding='utf-8-sig' 
destino.to_csv(ruta_destino_guardado, index=False, encoding='utf-8-sig')